# 05 — Combined results: source-weighted and ordinary pooled
Run [00](00_Data_Preparation.ipynb), then all four model notebooks: [RF](01_Random_Forest.ipynb), [XGBoost](02_XGBoost.ipynb), [GRU](03_GRU.ipynb), [LSTM](04_LSTM.ipynb).
This notebook reads saved predictions. **It does not retrain any model.**

The shared helper is generated by section 2 of notebook 00 and saved to Drive. Edit shared functions in notebook 00, not in a separate Python file.

## 1. Setup and read predictions

In [ ]:
%pip -q install "scikit-learn==1.6.1" "optuna>=4,<5" "xgboost>=2,<4" joblib

In [ ]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/KC_CO2_Modular_Notebooks")
sys.path.insert(0, str(PROJECT))
RUN_DIR = PROJECT / "runs" / "real_sessions_v1"
RUN_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from kc_common import *


In [ ]:
metadata = json.loads((RUN_DIR / "preparation.json").read_text())
frames = [pd.read_csv(RUN_DIR / "persistence_predictions.csv.gz")]
for name in ["RF", "XGBoost", "GRU", "LSTM"]:
    frames.append(pd.read_csv(RUN_DIR / name / "predictions.csv.gz"))
predictions = pd.concat(frames, ignore_index=True)
assert set(predictions.run_id) == {
    metadata["run_id"]
}, "Rerun models after changing preparation."
keys = ["Model", "Horizon_min", "Approach", "Split"]
# Each model must evaluate exactly the same rows and actual values as persistence.
for key, group in predictions.groupby(keys):
    baseline = predictions[
        (predictions.Model == "Persistence")
        & (predictions.Horizon_min == key[1])
        & (predictions.Split == key[3])
    ]
    a = group.sort_values("row_id")
    b = baseline.sort_values("row_id")
    assert a.row_id.is_unique and np.array_equal(a.row_id, b.row_id)
    assert np.allclose(a.actual_future, b.actual_future)
print("All model outputs use this preparation run and matching evaluation rows.")


## 2. What the two combined scores mean
**Source-weighted:** Local gets 50%, UPC 25%, Slovenia 25%, regardless of row counts. MAE = 0.5 × Local MAE + 0.25 × UPC MAE + 0.25 × Slovenia MAE. These are the earlier training and tuning weights.

**Ordinary pooled:** every evaluation row counts equally. Slovenia supplies most rows and therefore has the greatest influence. This is the earlier `ALL` result.

Weighted RMSE is the square root of the weighted average **squared** error, not the average of source RMSE values. Weighted R² uses a weighted overall mean and weighted sums of squares, not the average of source R² values. Weighted RMSE/R² are added here for review; earlier model selection used weighted MAE only.

In [ ]:
source_rows, pooled_rows, weighted_rows = [], [], []
for key, group in predictions.groupby(keys):
    labels = dict(zip(keys, key))
    for source, part in group.groupby("source"):
        source_rows.append(
            {
                **labels,
                "Source": source,
                "Rows": len(part),
                **metrics(part.actual_future, part.predicted_future),
            }
        )
    pooled_rows.append(
        {**labels, "Rows": len(group), **metrics(group.actual_future, group.predicted_future)}
    )
    weights = balanced_weights(group.source)
    weighted_rows.append(
        {
            **labels,
            "Rows": len(group),
            **metrics(group.actual_future, group.predicted_future, weights),
        }
    )
per_source = pd.DataFrame(source_rows)
pooled = pd.DataFrame(pooled_rows)
weighted = pd.DataFrame(weighted_rows)
for name, table in [
    ("per_source", per_source),
    ("pooled", pooled),
    ("source_weighted", weighted),
]:
    table.to_csv(RUN_DIR / f"{name}_results.csv", index=False)


## 3. Print source-weighted results for both horizons

In [ ]:
for horizon in (5, 10):
    print(f"{horizon}-minute SOURCE-WEIGHTED scores")
    display(weighted.query("Horizon_min == @horizon").sort_values(["Split", "MAE"]).round(3))


## 4. Print ordinary pooled results and individual dataset results

In [ ]:
for horizon in (5, 10):
    print(f"{horizon}-minute ORDINARY POOLED scores")
    display(pooled.query("Horizon_min == @horizon").sort_values(["Split", "MAE"]).round(3))
display(
    per_source.sort_values(["Horizon_min", "Split", "Model", "Approach", "Source"]).round(3)
)


## 5. Compare tree approaches across forecast horizons

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for (model, approach), group in weighted.query(
    "Split == 'Validation' and Model in ['RF', 'XGBoost']"
).groupby(["Model", "Approach"]):
    group = group.sort_values("Horizon_min")
    ax.plot(group.Horizon_min, group.MAE, marker="o", label=f"{model} {approach}")
ax.set(
    xticks=[5, 10],
    xlabel="Forecast horizon (minutes)",
    ylabel="Source-weighted validation MAE (ppm)",
    title="Direct and recursive tree forecasts",
)
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(RUN_DIR / "tree_methods_by_horizon.png", dpi=180)
plt.show()


## 6. Compare with persistence on the same rows
Positive improvement means lower MAE than persistence. A small error difference alone does not prove better ventilation decisions or faster ESP32 operation.

In [ ]:
baseline = weighted.query("Model == 'Persistence'")[["Horizon_min", "Split", "MAE"]].rename(
    columns={"MAE": "Persistence_MAE"}
)
comparison = weighted.query("Model != 'Persistence'").merge(
    baseline, on=["Horizon_min", "Split"]
)
comparison["MAE_improvement_ppm"] = comparison.Persistence_MAE - comparison.MAE
comparison["MAE_improvement_percent"] = (
    100 * comparison.MAE_improvement_ppm / comparison.Persistence_MAE
)
comparison.to_csv(RUN_DIR / "persistence_comparison.csv", index=False)
display(comparison.round(3))


## Interpreting the outcome
Use validation results to review selection; report test results separately. Previously examined test sessions are not new independent evidence. Direct RF is not assumed to win. Report recursive RF's RMSE advantage wherever the new run shows it. The neural five-minute runs are extensions and should not be described as previously measured results.
All model runs train jointly on the three sources. Dataset-specific output tables are not separately trained dataset-specific models.